# Refine Runs Export

This notebook scans the full `runs/` tree, preserves the directory structure, and copies only these files into a new refined folder:
- `config.json`
- `history.csv`
- `step_history.csv`
- `summary.json`

It skips anything inside `artifacts/` and `checkpoints/`.


In [6]:
from __future__ import annotations

import shutil
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

RUNS_ROOT = Path('/home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs').resolve()
REFINED_ROOT = Path('/home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs_refined').resolve()

FILES_TO_COPY = {'config.json', 'history.csv', 'step_history.csv', 'summary.json'}
SKIP_DIR_NAMES = {'artifacts', 'checkpoints'}

# Preview first with DRY_RUN = True. Set it to False to perform the copy.
DRY_RUN = False

# If True and DRY_RUN is False, existing REFINED_ROOT will be deleted first.
RESET_REFINED_ROOT = False

RUNS_ROOT, REFINED_ROOT


(PosixPath('/home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs'),
 PosixPath('/home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs_refined'))

In [7]:
def should_skip_path(path: Path, skip_dir_names: set[str]) -> bool:
    return any(part in skip_dir_names for part in path.parts)


def collect_candidate_files(
    runs_root: Path,
    files_to_copy: set[str],
    skip_dir_names: set[str],
) -> list[Path]:
    if not runs_root.exists():
        raise FileNotFoundError(f'Runs root does not exist: {runs_root}')

    files: list[Path] = []
    for path in runs_root.rglob('*'):
        if not path.is_file():
            continue
        if path.name not in files_to_copy:
            continue
        if should_skip_path(path.relative_to(runs_root), skip_dir_names):
            continue
        files.append(path)
    return sorted(files)


def build_inventory(files: list[Path], runs_root: Path) -> pd.DataFrame:
    rows = []
    for source_path in files:
        relative_path = source_path.relative_to(runs_root)
        parts = relative_path.parts
        rows.append(
            {
                'relative_path': str(relative_path),
                'group_dir': parts[0] if len(parts) > 0 else None,
                'run_dir': parts[1] if len(parts) > 1 else None,
                'file_name': source_path.name,
                'source_path': str(source_path),
                'size_bytes': source_path.stat().st_size,
            }
        )
    return pd.DataFrame(rows)


def reset_directory(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def copy_refined_tree(
    files: list[Path],
    runs_root: Path,
    refined_root: Path,
    dry_run: bool,
    reset_refined_root: bool,
) -> pd.DataFrame:
    operations = []
    if not dry_run:
        if reset_refined_root:
            reset_directory(refined_root)
        else:
            refined_root.mkdir(parents=True, exist_ok=True)

    for source_path in files:
        relative_path = source_path.relative_to(runs_root)
        destination_path = refined_root / relative_path
        operations.append(
            {
                'source_path': str(source_path),
                'destination_path': str(destination_path),
                'relative_path': str(relative_path),
                'file_name': source_path.name,
            }
        )
        if not dry_run:
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

    return pd.DataFrame(operations)


In [8]:
candidate_files = collect_candidate_files(RUNS_ROOT, FILES_TO_COPY, SKIP_DIR_NAMES)
inventory_df = build_inventory(candidate_files, RUNS_ROOT)

print(f'Runs root: {RUNS_ROOT}')
print(f'Refined root: {REFINED_ROOT}')
print(f'Candidate files found: {len(candidate_files)}')
print('Files by type:')
print(Counter(path.name for path in candidate_files))

if not inventory_df.empty:
    display(
        inventory_df.groupby(['group_dir', 'file_name']).size().rename('count').reset_index().sort_values(['group_dir', 'file_name'])
    )
    display(inventory_df.head(20))
else:
    print('No matching files found.')


Runs root: /home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs
Refined root: /home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs_refined
Candidate files found: 426
Files by type:
Counter({'config.json': 108, 'history.csv': 106, 'step_history.csv': 106, 'summary.json': 106})


,group_dir,file_name,count
0,cola_contraction,config.json,17
1,cola_contraction,history.csv,17
2,cola_contraction,step_history.csv,17
3,cola_contraction,summary.json,17
4,cola_reconstruction,config.json,17
5,cola_reconstruction,history.csv,17
6,cola_reconstruction,step_history.csv,17
7,cola_reconstruction,summary.json,17
8,csqa_reconstruction_nolearning,config.json,12
9,csqa_reconstruction_nolearning,history.csv,11


,relative_path,group_dir,run_dir,file_name,source_path,size_bytes
0,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,config.json,/home/pkunwar/characterize_ttlora/phases/ttlor...,2043
1,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,history.csv,/home/pkunwar/characterize_ttlora/phases/ttlor...,33549
2,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,step_history.csv,/home/pkunwar/characterize_ttlora/phases/ttlor...,4188312
3,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,summary.json,/home/pkunwar/characterize_ttlora/phases/ttlor...,25843
4,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,config.json,/home/pkunwar/characterize_ttlora/phases/ttlor...,2061
5,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,history.csv,/home/pkunwar/characterize_ttlora/phases/ttlor...,46237
6,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,step_history.csv,/home/pkunwar/characterize_ttlora/phases/ttlor...,5242021
7,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,summary.json,/home/pkunwar/characterize_ttlora/phases/ttlor...,28206
8,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,config.json,/home/pkunwar/characterize_ttlora/phases/ttlor...,2079
9,cola_contraction/ttcore_roberta_mrpc_cola_core...,cola_contraction,ttcore_roberta_mrpc_cola_core_study_cola_contr...,history.csv,/home/pkunwar/characterize_ttlora/phases/ttlor...,46435


In [9]:
copy_log_df = copy_refined_tree(
    candidate_files,
    runs_root=RUNS_ROOT,
    refined_root=REFINED_ROOT,
    dry_run=DRY_RUN,
    reset_refined_root=RESET_REFINED_ROOT,
)

if DRY_RUN:
    print('Dry run only. No files were copied.')
else:
    print(f'Copied {len(copy_log_df)} files into {REFINED_ROOT}')

display(copy_log_df.head(20))


Copied 426 files into /home/pkunwar/characterize_ttlora/phases/ttlora_core_count_study/runs_refined


,source_path,destination_path,relative_path,file_name
0,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,config.json
1,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,history.csv
2,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,step_history.csv
3,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,summary.json
4,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,config.json
5,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,history.csv
6,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,step_history.csv
7,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,summary.json
8,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,config.json
9,/home/pkunwar/characterize_ttlora/phases/ttlor...,/home/pkunwar/characterize_ttlora/phases/ttlor...,cola_contraction/ttcore_roberta_mrpc_cola_core...,history.csv
